In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import OptimConfig
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles_from_graph,
)
from vizopt.examples.sets import make_animals_graph, make_multiples_of_primes_graph
from vizopt.schedules import make_term_schedules

# Sets of multiples

In [ ]:
primes = [2, 3, 5]
G = make_multiples_of_primes_graph(primes)
elements = sorted(n for n in G.nodes if G.out_degree(n) == 0)

In [ ]:
n_iters = 8000
best_params = {
    "collision_delay": 0.27,
    "collision_ramp":  0.33,
    "exclusion_delay": 0.32,
    "exclusion_ramp":  0.47,
    "area_delay":      0.31,
    "area_ramp":       0.27,
    "perimeter_delay": 0.69,
    "perimeter_ramp":  0.45,
    "attraction_peak": 0.69,
    "attraction_ramp": 0.18,
}
term_schedules = make_term_schedules(best_params, n_iters)

In [ ]:
snapshot_cb = SnapshotCallback(every=50)

named_results, named_circles_out, history, problem = optimize_radially_convex_sets_and_circles_from_graph(
    G,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.3,
    weight_circle_collision=20.0,
    weight_set_attraction=5.0,
    term_schedules=term_schedules,
    optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    callback=snapshot_cb,
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
set_colors = ["tomato", "steelblue", "mediumorchid", "mediumseagreen"]
set_node_names = [f"multiples_of_{p}" for p in primes]
set_labels = [f"Multiples of {p}" for p in primes]

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color, label in zip(set_node_names, set_colors, set_labels):
    result = named_results[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=label)
    ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

for n in elements:
    r_orig = G.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_out[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, str(n), ha="center", va="center", fontsize=10, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.legend(loc="upper right", fontsize=11)
ax.set_title("Multiples of 2, 3, 5 among 1-11\nStar-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".", figsize=(7, 3))
plt.ylabel("Loss value")
plt.title("Optimization history")
plt.tight_layout()

In [ ]:
svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=5, size=600, history=history)
SVG(data=svg)

# Animals taxonomy

In [ ]:
G_a = make_animals_graph()
elements_a = sorted(n for n in G_a.nodes if G_a.out_degree(n) == 0)
set_names_a = [n for n in nx.topological_sort(G_a) if G_a.out_degree(n) > 0]

In [ ]:
named_results_a, named_circles_a, history_a, problem_a = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G_a,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.01,
        weight_circle_collision=100.0,
        weight_set_attraction=0.1,
        circle_collision_alpha=1.0,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

In [ ]:
history_a[-1]

In [ ]:
def boundary_label_position(center, radii, angles, member_circles, inset=1.01):
    """Boundary point with maximum clearance from member circles, pulled slightly inward."""
    center = np.asarray(center)
    bx = center[0] + radii * np.cos(angles)
    by = center[1] + radii * np.sin(angles)
    bp = np.stack([bx, by], axis=1)  # (K, 2)

    member_circles = np.asarray(member_circles)
    if member_circles.ndim == 2 and len(member_circles) > 0:
        dists = np.linalg.norm(bp[:, None, :] - member_circles[None, :, :2], axis=2)
        clearances = (dists - member_circles[None, :, 2]).min(axis=1)
    else:
        clearances = radii

    k = int(np.argmax(clearances))
    return center + inset * radii[k] * np.array([np.cos(angles[k]), np.sin(angles[k])])

In [ ]:
leaf_set_a = {n for n in G_a.nodes if G_a.out_degree(n) == 0}
colors_a = plt.cm.tab10(np.linspace(0, 0.9, len(set_names_a)))

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color in zip(set_names_a, colors_a):
    result = named_results_a[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2)
    #ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

    members = [n for n in nx.descendants(G_a, set_name) if n in leaf_set_a]
    member_circles = np.array([named_circles_a[n] for n in members])
    lx, ly = boundary_label_position(np.array([cx, cy]), radii, angles, member_circles)
    ax.text(lx, ly, set_name, ha="center", va="center", fontsize=9, fontweight="bold", color=color)

for n in elements_a:
    r_orig = G_a.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_a[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, n, ha="center", va="center", fontsize=9, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.set_title("Animal taxonomy — star-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()

# Label rectangles

Re-run the animals example with `label_rect_size` to reserve a floating label area at the top of each set boundary.

In [ ]:
named_results_l, named_circles_l, history_l, problem_l = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G_a,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.01,
        weight_circle_collision=100.0,
        weight_set_attraction=0.1,
        circle_collision_alpha=1.0,
        label_rect_size=(0.6, 0.15),
        weight_label_enclosure=20.0,
        weight_label_element_exclusion=20.0,
        weight_label_collision=10.0,
        weight_label_top=1.0,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

In [ ]:
from matplotlib.patches import FancyBboxPatch

colors_l = plt.cm.tab10(np.linspace(0, 0.9, len(set_names_a)))
plot_label_box = False

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color in zip(set_names_a, colors_l):
    result = named_results_l[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2)

    lx, ly = result["label_center"]
    hw, hh = 0.6, 0.15
    if plot_label_box:
        ax.add_patch(FancyBboxPatch(
            (lx - hw, ly - hh), 2 * hw, 2 * hh,
            boxstyle="round,pad=0.05", linewidth=1.2,
            edgecolor=color, facecolor=color, alpha=0.25,
        ))
    ax.text(lx, ly, set_name, ha="center", va="center",
            fontsize=8, fontweight="bold", color=color)

for n in elements_a:
    r_orig = G_a.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_l[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, n, ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.set_title("Animal taxonomy — label rectangles reserved at top of each set boundary")
ax.axis("off")
plt.tight_layout()